# 노드 재시작과 멱등성

부작용 뒤에 프로세스가 종료된 노드를 재개할 때 멱등 키가 중복 기록을 막는지 확인합니다.


## 실행 환경 준비

Python 3.12 커널에 이 절에서 사용할 패키지 버전을 고정합니다.


In [1]:
# 실습에 필요한 패키지를 같은 버전으로 설치합니다.
%pip install -q langgraph==1.2.11 langgraph-checkpoint-sqlite==3.0.3

Note: you may need to restart the kernel to use updated packages.


## 실제 중단과 재개 함수

작업 파일에 주문 번호를 기록한 직후 자식 프로세스를 종료합니다.

멱등성(idempotency)은 같은 연산 요청을 여러 번 받아도 외부 결과가 한 번 처리한 것과 같게 유지되는 성질입니다. 체크포인터는 노드 시작과 완료 사이에 프로세스가 종료되면 그 노드를 다시 실행할 수 있습니다. 파일 쓰기는 이미 끝났지만 노드 완료 체크포인트가 없으면 재개 시 같은 주문 번호를 다시 쓸 수 있습니다.

`interrupt_then_resume(strategy)`는 자식 프로세스가 외부 쓰기를 마친 직후 종료 신호 파일을 만들게 하고, 부모가 그 프로세스를 강제로 종료한 뒤 같은 SQLite와 전략으로 재개합니다. `operation_id="order-2306"`은 논리적 작업을 식별합니다. idempotent 전략은 이미 처리한 ID인지 검사해 두 번째 쓰기를 건너뜁니다.


In [2]:
# worker와 모드별 SQLite·로그 파일을 둘 실습 경로를 찾습니다.
import subprocess
import sys
import time
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
lab = next((p / "labs/part6/ch23-persistence" for p in candidates if (p / "labs/part6/ch23-persistence").is_dir()), None)
if lab is None:
    lab = next(p for p in candidates if p.name == "ch23-persistence")
worker = lab / "idempotency_worker.py"
run_dir = lab / "data" / "idempotency"
run_dir.mkdir(parents=True, exist_ok=True)

## 중단과 재개 함수

같은 SQLite 체크포인트와 전략 이름으로 새 프로세스를 실행합니다.


In [3]:
# 외부 쓰기 직후 자식 프로세스를 종료하고 같은 체크포인트에서 새 프로세스로 재개합니다.
def interrupt_then_resume(strategy: str) -> list[str]:
    db = run_dir / f"{strategy}.sqlite"
    log = run_dir / f"{strategy}.log"
    ready = run_dir / f"{strategy}.ready"
    for path in [db, log, ready]:
        path.unlink(missing_ok=True)
    first = subprocess.Popen([sys.executable, str(worker), str(db), str(log), str(ready), strategy, "first"])
    deadline = time.time() + 20
    while not ready.exists() and time.time() < deadline:
        time.sleep(0.02)
    assert ready.exists()
    first.kill()
    first.wait(timeout=5)
    second = subprocess.run(
        [sys.executable, str(worker), str(db), str(log), str(ready), strategy, "resume"],
        check=True, capture_output=True, text=True, timeout=20,
    )
    return log.read_text(encoding="utf-8").splitlines()

## Before: 멱등 키를 검사하지 않는 쓰기


In [4]:
# 중복 검사가 없는 naive 전략에서 같은 operation_id가 두 번 기록되는지 확인합니다.
naive_lines = interrupt_then_resume("naive")
print("naive log:", naive_lines)
print("same operation count:", naive_lines.count("order-2306"))
assert naive_lines.count("order-2306") == 2

naive log: ['order-2306', 'order-2306']
same operation count: 2


## After: operation_id로 중복 쓰기 차단

결제나 이메일 API가 멱등 키를 지원하면 호출자가 작업 ID를 보내고 서버가 이전 결과를 재사용하게 할 수 있습니다. 로컬에서 먼저 조회한 뒤 쓰는 방식은 두 프로세스가 동시에 통과할 수 있으므로 운영 저장소에서는 고유 제약이나 원자적 삽입을 사용해야 합니다.

재시도는 실패한 호출을 다시 시도하는 정책이고 멱등성은 반복 호출이 중복 효과를 만들지 않게 하는 성질입니다. 재시도 횟수를 1로 줄여도 종료 시점의 불확실성은 남습니다. 이번 로그 한 줄은 실습 worker의 중복 차단을 확인하며 외부 서비스 전체의 멱등성을 보장하지 않습니다.


In [5]:
# 처리된 operation_id를 검사하는 전략에서 로컬 쓰기가 한 줄만 남는지 확인합니다.
idempotent_lines = interrupt_then_resume("idempotent")
print("idempotent log:", idempotent_lines)
print("same operation count:", idempotent_lines.count("order-2306"))
assert idempotent_lines.count("order-2306") == 1

idempotent log: ['order-2306']
same operation count: 1
